In [41]:
model = "llama3.2:1B"

#### Task 1: Create a Simple Chain for Summarization


**Objective:**

Build a LangChain chain that can summarize a given text.

**Task Description:**

- Create a llm chain using with a Ollama model.
- Define a prompt template for summarization. The summary should be only one sentence.
- Run the chain with a sample text and print the summary.
- Add model output streaming.
- Run the chain with streaming with a sample text and print the summary.


**Hint: The five messages in LangChain:**


- SystemMessage: corresponds to system role
- HumanMessage: corresponds to user role
- AIMessage: corresponds to assistant role
- AIMessageChunk: corresponds to assistant role, used for streaming responses
- ToolMessage: corresponds to tool role


[More Information](https://python.langchain.com/docs/concepts/messages/)

**Useful links:**

- [How To Prompt Template 1](https://python.langchain.com/v0.2/docs/tutorials/extraction/#the-extractor)
- [How To Prompt Template 2](https://python.langchain.com/v0.2/api_reference/core/prompts/langchain_core.prompts.chat.ChatPromptTemplate.html#langchain_core.prompts.chat.ChatPromptTemplate)
- [How To LCEL Chains 1](https://python.langchain.com/v0.2/docs/concepts/#langchain-expression-language-lcel)
- [How To LCEL Chains 2](https://python.langchain.com/v0.2/docs/versions/migrating_chains/llm_chain/#lcel)
- [How To Chain Streaming](https://python.langchain.com/v0.2/docs/concepts/#streaming)

In [ ]:
from langchain_ollama.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

# modell laden 
model = "llama3.2:1b"
llm = ChatOllama(model=model)

# prompt template definieren
summarization_prompt = ChatPromptTemplate.from_messages([
    ("system", "Du bist ein hilfreicher Assistent. Fasse den folgenden Text in genau einem Satz auf Deutsch zusammen."),
    ("human", "{text}"),
])

# chain erstellen
# '|' verbindet den rompt direkt mit dem sprachmodell
summarization_chain = summarization_prompt | llm

# Sample text
text = """Over the last decade, deep learning has evolved massively to process and generate unstructured data like text, images, and video. 
These advanced AI models have gained popularity in various industries, and include large language models (LLMs). 
There is currently a significant level of fanfare in both the media and the industry surrounding AI,
and there’s a fair case to be made that Artificial Intelligence (AI), with these advancements,
is about to have a wide-ranging and major impact on businesses, societies, and individuals alike.
This is driven by numerous factors, including advancements in technology, high-profile applications, 
and the potential for transfor- mative impacts across multiple sectors."""

# Invoke
print("--- Ergebnis (Normaler Aufruf): ---")
summary = summarization_chain.invoke({"text": text})
print(summary.content)

print("\n" + "-"*30 + "\n")

# streaming aufruf
print("--- Ergebnis (Streaming): ---")
for chunk in summarization_chain.stream({"text": text}):
    print(chunk.content, end="", flush=True)

--- Ergebnis (Normaler Aufruf): ---
Während die letzten Decade der maschinellem Lernprozess massiv zum Verständnis und zur Generierung unstructured Daten wie Texten, Bildern und Videos gewachsen ist.

------------------------------

--- Ergebnis (Streaming): ---
Deep learning hat sich in den letzten zehn Jahren extrem intensiv entwickelt und kann nun unstrukturierte Daten wie Text, Bilder und Videos verarbeiten und generieren. Diese fortschrittlichen AI-Modelle haben große Erfolge in verschiedenen Branchen gefeiert, einschließlich großen Sprachmodellen (LLMs).

In [43]:
# Stream the chain output
for chunk in summarization_chain.stream({"text": text}):
    print(chunk.content, end="", flush=True)

Die Entwicklung des Deep-Lernings in den letzten Jahren hat sich massiv verändert und kann nun für Texte, Bilder und Videos verwendet werden, was zu einer enormen Entwicklung führt. Diese fortschrittlichen AI-Modelle sind besonders beliebt in verschiedenen Branchen und einschließlich großen Sprachmodellen (LLMs).

#### Task 2: Chain with Tool Usage (Simple Math Tool)

**Objective:**

Create a LangChain chain that uses a simple math tool to perform calculations.

**Task Description:**

- Define a function as tool which multiplies two integer values and return the result.
- Create a chain
- Print the result of the calculation.

**Useful links:**

- [How To Tools](https://python.langchain.com/v0.2/docs/how_to/tools_chain/#create-a-tool)
- [How To Tools in Chains](https://python.langchain.com/v0.2/docs/how_to/tools_chain/#chains)
- [How To Tool Calling](https://python.langchain.com/v0.2/docs/concepts/#functiontool-calling)
- [How To Chain and Call Tools with Ollama](https://python.langchain.com/v0.2/docs/integrations/chat/ollama/)

In [59]:
from langchain_core.tools import tool

# Create custom tool
@tool
def multiply(first_int: int, second_int: int) -> int:
    """Multiplies two integer values and returns the result."""
    return first_int * second_int


print(multiply.name)
print(multiply.description)
print(multiply.args) 

# Invoke custom tool
multiply.invoke({"first_int": 4, "second_int": 5})

multiply
Multiplies two integer values and returns the result.
{'first_int': {'title': 'First Int', 'type': 'integer'}, 'second_int': {'title': 'Second Int', 'type': 'integer'}}


20

In [60]:
# Wir binden das Tool an das Modell
llm_with_tools = llm.bind_tools([multiply])

# Aufruf
msg = llm_with_tools.invoke("whats 5 times forty two")
print(msg.tool_calls)

[{'name': 'multiply', 'args': {'first_int': '42', 'second_int': '5'}, 'id': '4bb4a646-4469-4df0-afd6-38a4feba6f65', 'type': 'tool_call'}]


In [47]:
# Create the chain: Extrahiere Argumente und führe das Tool aus
# Wir prüfen, ob tool_calls vorhanden sind, um Fehler zu vermeiden
chain_with_tools = llm_with_tools | (lambda x: multiply.invoke(x.tool_calls[0]["args"]) if x.tool_calls else "Kein Tool-Aufruf erkannt")

# Run chain
result = chain_with_tools.invoke("whats 5 times forty two")
print(f"Ergebnis: {result}")

Ergebnis: 210


#### Task 3: Agent with Tool Usage (Two Tools)

**Objective:** 

Create a LangChain agent that uses two tools to perform tasks.

**Task Description:**

- Define the system prompt.
- Define tools.
- Create an Agent using the Ollama model, system-prompt and tools.
- Run the agent with a prompt that requires one or both tools.
- Observe how the agent uses the tools to complete the task.


**Useful links:**

- [How To](https://docs.langchain.com/oss/javascript/langchain/agents)

In [ ]:
from langchain.agents import create_agent
from langchain_ollama.chat_models import ChatOllama
from langchain_core.tools import tool

# Load the Ollama model 
llm = ChatOllama(model=model)

# System-Prompt definieren
agent_prompt = "Du bist ein hilfreicher KI-Assistent für Mathematik. Nutze die bereitgestellten Tools, um die Aufgaben des Nutzers Schritt für Schritt zu lösen."

# Custom math tools
@tool
def add(first_int: int, second_int: int) -> int:
    """Add two integers."""
    return first_int + second_int

@tool
def exponentiate(base: int, exponent: int) -> int:
    """Exponentiate the base to the exponent power."""
    return base**exponent

tools = [add, exponentiate]

In [ ]:

agent_with_tools = create_agent(llm, tools)

In [62]:
# promt übergeben als erste "system" nachricht, 
response = agent_with_tools.invoke(
    {"messages": [
        {"role": "system", "content": agent_prompt},
        {"role": "user", "content": "First take 3 to the power of five and afterwards add 12?"}
    ]}
)

# ausgabe
for message in response["messages"]:
    print(message.content)

Du bist ein hilfreicher KI-Assistent für Mathematik. Nutze die bereitgestellten Tools, um die Aufgaben des Nutzers Schritt für Schritt zu lösen.
First take 3 to the power of five and afterwards add 12?

8
Die Antwort ist: 3^5 + 12 = 243 + 12 = 255
